<img src="https://github.com/JuliaLang/julia/raw/master/doc/src/assets/logo.svg" width="72" alt="Julia logo" /> <img src="https://avatars.githubusercontent.com/u/7346142?s=200&v=4" width="72" alt="KernelAbstractions logo" />

In [ ]:
# Pick your hardware: :CUDA (NVIDIA), :AMDGPU (AMD), :Metal (Apple), or :CPU
DEV = :CUDA


# 1. Setup

In julia, we use environments to define a package setup for a specific projects. Without an environment, we risk competing dependencies between different projects. This folder contains a `Project.toml` and a `Manifest.toml`, which together define a Julia project environment. Set `DEV` in the **second cell** (above), then run the cells below: `Pkg.activate(@__DIR__)` makes this environment active instead of your global one (`@__DIR__` is the directory containing the notebook). The `Pkg.add` call installs packages for your `DEV` and records them in this `Project.toml`. Activation lasts for the current Julia session, so these cells have to be rerun after a kernel restart.

In [ ]:
using Pkg
Pkg.activate(@__DIR__)

In [ ]:
packages = ["BenchmarkTools", "KernelAbstractions", "Plots"]
if DEV != :CPU
    push!(packages, string(DEV))
end
Pkg.add(packages)


In [ ]:
using BenchmarkTools, LinearAlgebra, KernelAbstractions, Plots


In [ ]:
Pkg.status()

## 1.1. Selecting a backend

Each vendor has its own Julia package (CUDA.jl, AMDGPU.jl, Metal.jl). `Pkg.add` uses `string(DEV)` when `DEV != :CPU`. The cell below runs `@eval using $(DEV)` and sets `ArrayKA` and `Backend`.


In [ ]:
if DEV != :CPU
    @eval using $(DEV)
end

if DEV == :CUDA
    ArrayKA = CUDA.CuArray
    Backend = CUDA.CUDABackend()
elseif DEV == :AMDGPU
    ArrayKA = AMDGPU.ROCArray
    Backend = AMDGPU.ROCBackend()
elseif DEV == :Metal
    ArrayKA = Metal.MtlArray
    Backend = Metal.MetalBackend()
elseif DEV == :CPU
    ArrayKA = Array
    Backend = CPU()
else
    error("Unknown DEV = $DEV. Use :CUDA, :AMDGPU, :Metal, or :CPU.")
end


# 2. Vector addition

We start with elementwise vector addition, $\vec{c} = \vec{a} + \vec{b}$. Let's allocate and initialize the vectors.

In [ ]:
# inputs a, b and output c in host memory
vector_size = 1024
a = rand(1:4, vector_size)
b = rand(1:4, vector_size)
c = zeros(Int, vector_size)

a

## 2.1. CPU vector addition

Julia can form the sum in two ways. **`c = a + b`** allocates a **new** vector and returns it (the old binding for `c` is replaced). **`c .= a .+ b`** writes into the **existing** storage of `c` and does not allocate a new vector for the result.

In [ ]:
c = a + b
c

In [ ]:
c .= a .+ b
c

The next cell benchmarks both forms. In each `Trial`, **`allocs`** is the number of heap allocations per call (`a + b` allocates a new vector every time; `c .= a .+ b` reuses `c`). You should see more allocations and often more garbage-collection time for `a + b`, because Julia must reserve fresh storage for the result on every call even though the elementwise addition is the same. Timing and allocation both matter on the GPU, where we pre-allocate `dc` instead of repeating `dc = da + db` in a loop.

In [ ]:
t_alloc = @benchmark c = a + b
t_inplace = @benchmark (c .= a .+ b)

println("heap allocations per call: a + b → ", t_alloc.allocs, ",  c .= a .+ b → ", t_inplace.allocs)
display(t_alloc)
display(t_inplace)

## 2.2. GPU arrays and array operations

To do the same on the GPU, we first allocate device arrays and copy the inputs over. The addition writes into pre-allocated `dc`, as `c .= a .+ b` did on the CPU in §2.1—not `dc = da + db`, which would allocate a new device vector every time.

In [ ]:
# allocate reserves uninitialized device memory; copyto! copies host data into it

da = allocate(Backend, eltype(a), size(a)...)
copyto!(da, a)
db = allocate(Backend, eltype(b), size(b)...)
copyto!(db, b)

# KernelAbstractions.zeros allocates a zero-initialized device array
dc = KernelAbstractions.zeros(Backend, Int, size(a)...)

# displaying a device array copies it to the host; note the array type
da

`+` works on device arrays as well, on every backend: (Note that we refer to GPU as device and CPU as host.)

In [ ]:
dc .= da .+ db


## 2.3. Scaling of `+`

Elementwise addition on the CPU (`c .= a .+ b`) and on the device (`dc .= da .+ db`) launch the same kind of work. On the device the call returns once the work has been sent to the GPU, not when the addition has finished. The next cell times `dc .= da .+ db` twice: once as the call stands, and once followed by `KernelAbstractions.synchronize`, which blocks the host until the device is done.

In [ ]:
# submission only: returns before the GPU is done
@btime $dc .= $da .+ $db

# submission + wait: the time until the addition has finished
@btime begin
    $dc .= $da .+ $db
    KernelAbstractions.synchronize($Backend)
end

Only the second measurement includes the addition. For this vector of 1024 elements both numbers are mostly the cost of sending the work.

The next cell repeats that synchronized device measurement and `z .= x .+ y` on the CPU at lengths `2^3`, `2^5`, …, `2^21`. Each iteration builds fresh host vectors `x` and `y` with `rand` on the **CPU**, and **allocates** device arrays and **copies** the host data into them.

In [ ]:
sizes = 2 .^ (3:2:21)
t_cpu_plus= []
t_dev_plus = []
for n in sizes
    x, y = rand(Float32, n), rand(Float32, n)
    z = similar(x)
    push!(t_cpu_plus, @belapsed $z .= $x .+ $y)
    dx = allocate(Backend, Float32, n)
    copyto!(dx, x)
    dy = allocate(Backend, Float32, n)
    copyto!(dy, y)
    dz = KernelAbstractions.zeros(Backend, Float32, n)
    push!(t_dev_plus, @belapsed begin
        $dz .= $dx .+ $dy
        KernelAbstractions.synchronize($Backend)
    end)
end

tlabel(v) = begin
    x, u = v < 1e-6 ? (v * 1e9, "ns") : v < 1e-3 ? (v * 1e6, "µs") : v < 1 ? (v * 1e3, "ms") : (v, "s")
    r = round(x; sigdigits = 2)
    string(isinteger(r) ? Int(r) : r, " ", u)
end

Plots.plot(sizes, Float64.([t_cpu_plus t_dev_plus]);
    label = ["CPU .+" "device .+"],
    xlabel = "length", ylabel = "time",
    xscale = :log2, yscale = :log10,
    xticks = (sizes, string.(sizes)), xrotation = 45,
    yformatter = tlabel, marker = :circle, legend = :topleft)

On the left of the figure the device time barely changes with length. Launching the broadcast costs as much as the addition, so device `.+` is slower than CPU `.+`. Once the vector is long enough that the addition dominates the launch, device `.+` is faster, and the advantage grows with the length.

<img src="img/CPUvsGPU.png" alt="CPU versus GPU: a few large cores with control and cache on the left, a wide grid of small arithmetic units on the right" style="max-width:680px; width:100%; height:auto; display:block;" />

A CPU (left) has a few powerful cores. Most of its area goes to control logic and cache, with comparatively few arithmetic units (green). It is designed for low latency on a small number of complex instruction streams, such as the serial loop behind `c .= a .+ b`.

A GPU (right) devotes that area to many simple arithmetic units. It is designed for throughput, applying the same instruction to many data elements at once. Device `+` has this structure: one addition per element, across those units.

## 2.4. Host loops over device arrays

`da`, `db`, and `dc` now live in GPU memory. What happens if the host indexes them one element at a time?

In [ ]:
try
    for i in eachindex(dc)
        dc[i] = da[i] + db[i]
    end
catch e
    showerror(stdout, e)
end

The loop fails with a *scalar indexing* error. It runs on the host, so every `da[i]`, `db[i]`, and `dc[i]` would be a separate, synchronous transfer between device and host memory: about 3000 transfers for 1024 elements. The result would be correct but very slow, so GPU array packages disallow scalar indexing by default. (With `DEV = :CPU`, `da` is an ordinary `Array` and the loop runs.)

`dc .= da .+ db` in §2.2 works because multiple dispatch selects a GPU broadcast that launches a kernel into pre-allocated storage. The same holds for `da + db`, `sum`, `maximum`, `map`, and `reduce` on device arrays.

To run our own loop on the device, we write a **kernel**: a function executed on the GPU, with all reads and writes in device memory. Custom kernels are needed when a computation cannot be expressed with the built-in array operations, but whenever possible you should use built-in operations as those will be faster. The purpose of this tutorial is just educational: to familiarize yourself with how a GPU works.

## 2.5. Writing your first kernel

Tutorial kernels use the `my_*_gpu_kernel!` prefix (for example `my_vadd_gpu_kernel!`) so they are not confused with Base or BLAS.

Instead of one loop that visits every element, a kernel describes what a single loop iteration does, and the GPU runs many instances of it concurrently. Each instance is called a work item (a *thread* in CUDA terminology), and work items are grouped into workgroups (CUDA: *thread blocks*).

One step of `c .= a .+ b`, as a KernelAbstractions kernel:

In [ ]:
@kernel function my_vadd_gpu_kernel!(c, a, b)
    i = @index(Local)
    c[i] = a[i] + b[i]
end

`@kernel` marks the function as a kernel, i.e. the code executed by one work item. `@index(Local)` returns the index of the work item within its workgroup, starting at 1. Since we launch a single workgroup covering the whole vector, this is also the element index; section 2.6 shows why that breaks down for longer vectors. Kernels do not return values; they write their results into an array argument, here `c`.

A launch takes two steps. First the kernel is instantiated for a backend and a workgroup size, the number of work items per workgroup. This does not run anything yet:

In [ ]:
kernel! = my_vadd_gpu_kernel!(Backend, length(da))

Then it is called with its arguments and an `ndrange`, the total number of work items:

In [ ]:
kernel!(dc, da, db, ndrange=length(da))

A kernel launch, like `dc .= da .+ db`, returns once the work has been submitted. `KernelAbstractions.synchronize(Backend)` is the same wait as in the timings of section 2.3, and we call it here before reading the result. This is different from `@synchronize`, which appears later inside kernels and makes the work items of a workgroup wait for one another.

In [ ]:
KernelAbstractions.synchronize(Backend)

# copy the result back to the CPU and compare with the CPU answer
Array(dc) == a + b

<div style="border-left: 4px solid #5c7cfa; background: rgba(92, 124, 250, 0.08); padding: 0.4em 1.2em 0.6em; border-radius: 0 6px 6px 0; margin: 1em 0;">

## Exercise 1: SAXPY

SAXPY (single-precision *a x plus y*) is the BLAS level-1 operation `y = α * x + y`. Write `my_saxpy_gpu_kernel!` in the style of `my_vadd_gpu_kernel!`, with one work item per element and `@index(Local)`, since we again launch a single workgroup. Note that `y` is both input and output.

Write one line of code at each `# TODO` in both cells. The second cell copies `x` and `y` to the device; you build the launcher, launch it, and synchronize. It should print `true`. A solution is in `proposed-solution/exercise_1_saxpy.ipynb`.

</div>

In [ ]:
@kernel function my_saxpy_gpu_kernel!(y, α, x)
    # TODO 1: this work item's index i from @index(Local)

    # TODO 2: SAXPY update at index i

end

In [ ]:
α = 2.0f0
x = rand(Float32, 1024)
y = rand(Float32, 1024)

dx = allocate(Backend, Float32, length(x))
copyto!(dx, x)
dy = allocate(Backend, Float32, length(y))
copyto!(dy, y)

# TODO 3: build saxpy! for one workgroup of length 1024

# TODO 4: launch saxpy! on dy, α, and dx

KernelAbstractions.synchronize(Backend)

Array(dy) ≈ α .* x .+ y

## 2.6. Execution hierarchy and global indexing

<img src="img/threadsGPU.png" alt="GPU execution hierarchy: threads inside a thread block with shared memory, blocks gathered into clusters, and a grid over global memory" style="max-width:385px; width:100%; height:auto; display:block;" />

A kernel launch is organized in three levels, which KernelAbstractions and CUDA name differently:

| KernelAbstractions | NVIDIA / CUDA | Runs on | Memory at this level |
|---|---|---|---|
| work item | thread | one SIMD lane; lanes execute in groups of 32 (*warps*) | private registers |
| workgroup | thread block | one streaming multiprocessor (SM) | shared memory, visible to the workgroup |
| all work items (`ndrange`) | grid | the whole GPU | global memory, visible to all work items |

A workgroup always runs on a single SM (small hardware processor); a GPU has many SMs, but a workgroup is never split across them. Shared memory sits on the SM next to the arithmetic units and lets the work items of a workgroup exchange data quickly. `my_vadd_gpu_kernel!` has no use for it, since each work item touches only its own element; the matrix multiplication notebook does. Global memory, where `da`, `db`, and `dc` live, is visible to every work item but much slower.

The middle of the picture shows thread block clusters; we do not use them here.

Our first launch put all 1024 work items into one workgroup, which runs on one SM. What happens if the vector is ten times longer and we launch it the same way?

In [ ]:
vector_size = 10240
a = rand(1:4, vector_size)
b = rand(1:4, vector_size)

da = allocate(Backend, eltype(a), size(a)...)
copyto!(da, a)
db = allocate(Backend, eltype(b), size(b)...)
copyto!(db, b)
dc = KernelAbstractions.zeros(Backend, Int, vector_size)

# one workgroup of 10240 work items
kernel! = my_vadd_gpu_kernel!(Backend, length(da))
try
    kernel!(dc, da, db, ndrange=length(da))
catch e
    showerror(stdout, e)
end

This launch fails: the hardware limits the size of a workgroup, to 1024 work items on current NVIDIA and AMD GPUs. (The CPU backend's limit is larger, 4096 on our test machine, but still below 10240.)

In [ ]:
max_workgroupsize = 1024

The fix is to split the work into ten workgroups of 1024 (`cld(10240, 1024) = 10`), which the GPU distributes over its SMs. `@index(Local)` is then no longer sufficient: it restarts at 1 in every workgroup, so all ten workgroups would write to `c[1:1024]`. The index has to count across workgroups,

`i = @index(Local) + (@index(Group) - 1) * workgroupsize`,

where `@index(Group)` is the number of the workgroup (1 to 10) and `(@index(Group) - 1) * workgroupsize` is the number of elements covered by the preceding workgroups. KernelAbstractions provides this index directly as `@index(Global)`. With a single workgroup, `@index(Global)` and `@index(Local)` coincide, which is why the first kernel was correct.

In [ ]:
@kernel function my_vadd_gpu_kernel!(c, a, b)
    i = @index(Global)
    if i <= length(c)
        c[i] = a[i] + b[i]
    end
end

The `if` excludes work items beyond the end of the vector. 10240 divides evenly into workgroups of 1024, but a length such as 10000 would leave the last workgroup partly empty. KernelAbstractions already skips those work items; the check makes the assumption explicit.

We now launch with workgroups of 1024. `ndrange` still covers all 10240 elements, so KernelAbstractions creates ten workgroups:

In [ ]:
kernel! = my_vadd_gpu_kernel!(Backend, max_workgroupsize)
kernel!(dc, da, db, ndrange=length(da))
KernelAbstractions.synchronize(Backend)

Array(dc) == a + b

## 2.7. Timing `my_vadd_gpu_kernel!`

`my_vadd_gpu_kernel!` now uses `@index(Global)`, with a workgroup of at most `max_workgroupsize` and `ndrange` equal to the vector length, so the whole vector is covered at every length in the sweep, including those past 1024. The timings from section 2.3 are still in `t_cpu_plus` and `t_dev_plus`. The next cell times this kernel at the same lengths and adds it to the figure. Each device measurement waits with `synchronize`.

In [ ]:
t_dev_vadd = []
for n in sizes
    x, y = rand(Float32, n), rand(Float32, n)
    dx = allocate(Backend, Float32, n)
    copyto!(dx, x)
    dy = allocate(Backend, Float32, n)
    copyto!(dy, y)
    w = min(n, max_workgroupsize)
    dz, k! = similar(dx), my_vadd_gpu_kernel!(Backend, w)
    push!(t_dev_vadd, @belapsed begin
        $k!($dz, $dx, $dy; ndrange = $n)
        KernelAbstractions.synchronize($Backend)
    end)
end

Plots.plot(sizes, ([t_cpu_plus t_dev_plus t_dev_vadd]);
    label = ["CPU .+" "device .+" "my_vadd_gpu_kernel!"],
    xlabel = "length", ylabel = "time",
    xscale = :log2, yscale = :log10,
    xticks = (sizes, string.(sizes)), xrotation = 45,
    yformatter = tlabel, marker = :circle, legend = :topleft)

`my_vadd_gpu_kernel!` lies next to device `.+`: both perform the same addition, and both pay for the launch. On the long vectors both pull ahead of CPU `.+`.

## 2.8. Race conditions in global memory

Work items in different workgroups run concurrently and in no guaranteed order. This is harmless as long as each work item writes its own element, as in `my_vadd_gpu_kernel!`. Consider instead summing a vector of 10240 ones by having every work item add its element to `s[1]`:

In [ ]:
@kernel function my_racy_sum_gpu_kernel!(s, x)
    i = @index(Global)
    s[1] += x[i]
end

In [ ]:
dx = KernelAbstractions.ones(Backend, Int, 10240)
ds = KernelAbstractions.zeros(Backend, Int, 1)

my_racy_sum_gpu! = my_racy_sum_gpu_kernel!(Backend, max_workgroupsize)
my_racy_sum_gpu!(ds, dx, ndrange=length(dx))
KernelAbstractions.synchronize(Backend)

Array(ds)[1]

The result should be 10240; on a GPU backend you often get a much smaller value (for example 1). The statement `s[1] += x[i]` consists of a read of `s[1]`, an addition, and a write. Many work items read `s[1]` before any of them has written it back, so they all add to the same old value and overwrite each other's results. In such a race condition the result depends on the timing of the work items. On the CPU backend the same kernel happened to return 10240, so a passing test does not show that a kernel is free of races.

Two work items should therefore never write to the same location without coordination. For a sum, the built-in function takes care of this:

In [ ]:
sum(dx)

Kernels can also use atomic operations, which perform the read, the addition, and the write as one indivisible step. They are correct but slow when many work items update the same location, because the updates are serialized. The best solution is to ensure every location is only written to by a single thread while no other is reading.

# 3. Summary

- A portable `Backend`; moving data with `allocate`, `copyto!`, and `KernelAbstractions.zeros`
- Why a host loop over device arrays fails, and why built-in operations such as `+` still work
- Writing and launching a kernel, and how device `.+` and `my_vadd_gpu_kernel!` compare with CPU `.+`
- The execution hierarchy: work items, workgroups, and streaming multiprocessors
- Indexing across workgroups with `@index(Global)`
- Race conditions in global memory

Next, **`matrix_multiplication.ipynb`** covers 2D indexing, naive and shared-memory kernels for transpose and matrix multiplication, and why memory access dominates their performance.